#### **Download & Clean**

In [ ]:
import pandas as pd

#### **1. EIA sources**

Weekly series indexed by the Friday week-ending date. The imports file carries two metadata
rows before the header. The target is the first difference of stocks.

Only the WTI front month (M1) is kept in this first version. The M1–M2 calendar spread is
left out on purpose: under the theory of storage the stock level drives the spread rather
than the reverse, so the relationship is contemporaneous more than predictive. M2 stays on
disk for H3 and can be reassessed as an enrichment variable once the baseline holds.

In [6]:
src = "Dataset/IEA/"

# "Date" is the Friday week-ending date. The imports file has 2 metadata rows before the header.
stocks = pd.read_csv(src + "weekly_us_ending_stocks_excluding_spr.csv", parse_dates=["Date"])
production = pd.read_csv(src + "weekly_us_field_production.csv", parse_dates=["Date"])
refinery = pd.read_csv(src + "weekly_us_percent_utilization_of_refinery.csv", parse_dates=["Date"])
net_input = pd.read_csv(src + "weekly_us_refiner_net_input.csv", parse_dates=["Date"])
imports = pd.read_csv(src + "weekly_us_commercial_imports_excluding_spr.csv", parse_dates=["Date"])
wti_m1 = pd.read_csv(src + "weekly_WTI_front_month.csv", parse_dates=["Date"])

stocks.columns = ["week_ended", "crude_stocks"]
production.columns = ["week_ended", "production"]
refinery.columns = ["week_ended", "refinery_util"]
net_input.columns = ["week_ended", "refiner_net_input"]
imports.columns = ["week_ended", "commercial_imports"]
wti_m1.columns = ["week_ended", "wti_m1"]

eia = stocks
for df in [production, refinery, net_input, imports, wti_m1]:
    eia = eia.merge(df, on="week_ended", how="left")

eia["target_stock_change"] = eia["crude_stocks"].diff()  # target: weekly stock change

eia.head()

,week_ended,crude_stocks,production,refinery_util,refiner_net_input,commercial_imports,wti_m1,target_stock_change
0,1982-08-20,338764,NaN,NaN,11722,3459,NaN,NaN
1,1982-08-27,336138,NaN,NaN,11918,3354,NaN,-2626.0
2,1982-09-24,335586,NaN,NaN,12375,3494,NaN,-552.0
3,1982-10-01,334786,NaN,NaN,12303,2954,NaN,-800.0
4,1982-10-08,335260,NaN,NaN,12062,3008,NaN,474.0


#### **2. Reference date**

The Wednesday release (Friday + 5 days) becomes each row's reference date and the alignment
key for the explanatory variables.


In [7]:
# WPSR is released the Wednesday after the measured week (+5 days).
eia["publication_date"] = eia["week_ended"] + pd.Timedelta(days=5)

eia = eia[["publication_date", "week_ended", "target_stock_change", "crude_stocks",
           "production", "refinery_util", "refiner_net_input", "commercial_imports", "wti_m1"]]
eia.tail()

,publication_date,week_ended,target_stock_change,crude_stocks,production,refinery_util,refiner_net_input,commercial_imports,wti_m1
2278,2026-06-03,2026-05-29,-7974.0,433712,13707.0,94.7,16881,6397,87.36
2279,2026-06-10,2026-06-05,-7227.0,426485,13799.0,95.3,16962,5888,90.54
2280,2026-06-17,2026-06-12,-8263.0,418222,13806.0,96.7,17192,5134,84.88
2281,2026-06-24,2026-06-19,-6088.0,412134,13819.0,96.1,17111,5570,77.54
2282,2026-07-01,2026-06-26,-3775.0,408359,13810.0,96.6,17196,5279,69.23


#### **3. GPR aggregation**

The GPR is a daily index, while the target and every other series live on the weekly EIA
calendar. It must therefore be reduced to a single value per reference week before it can be
merged. Two choices define this reduction: which days enter each weekly value, and how they
are combined.

**Which days.** For a given Wednesday release, the window runs from the previous Thursday to
the Tuesday immediately before it (six calendar days). It ends the day before the release so
that no information from the announcement day, or later, can enter a feature used to predict
that same announcement — this is what rules out look-ahead bias. The window also captures the
most recent geopolitical conditions actually known to a forecaster standing just before the
release.

**How.** Daily values inside the window are averaged. The mean summarises the week's
geopolitical climate and smooths the day-to-day noise of the index, rather than depending on
a single arbitrary day.

The same window and the same mean are applied to the three sub-indices — overall (`GPRD`),
acts (`GPRD_ACT`) and threats (`GPRD_THREAT`) — yielding three weekly features aligned on the
publication date.


In [ ]:
gpr = pd.read_csv("Dataset/GPR/data_gpr_daily_recent.csv",
                  usecols=["DAY", "GPRD", "GPRD_ACT", "GPRD_THREAT"], parse_dates=["DAY"])
gpr = gpr.set_index("DAY").sort_index()

# Window: previous Thursday to Tuesday (day before release); nothing from the release day or later leaks into the features.
gpr_weekly = []
for pub in eia["publication_date"]:
    window = gpr.loc[pub - pd.Timedelta(days=6):pub - pd.Timedelta(days=1)]
    gpr_weekly.append(window.mean())

gpr_weekly = pd.DataFrame(gpr_weekly)
gpr_weekly.columns = ["gprd", "gprd_act", "gprd_threat"]
gpr_weekly["publication_date"] = eia["publication_date"].values

eia = eia.merge(gpr_weekly, on="publication_date", how="left")
eia.tail()

,publication_date,week_ended,target_stock_change,crude_stocks,production,refinery_util,refiner_net_input,commercial_imports,wti_m1,gprd,gprd_act,gprd_threat
2278,2026-06-03,2026-05-29,-7974.0,433712,13707.0,94.7,16881,6397,87.36,179.123333,184.291667,208.438333
2279,2026-06-10,2026-06-05,-7227.0,426485,13799.0,95.3,16962,5888,90.54,189.318333,211.383333,219.511667
2280,2026-06-17,2026-06-12,-8263.0,418222,13806.0,96.7,17192,5134,84.88,222.936667,251.953333,274.013333
2281,2026-06-24,2026-06-19,-6088.0,412134,13819.0,96.1,17111,5570,77.54,204.183333,174.561667,266.823333
2282,2026-07-01,2026-06-26,-3775.0,408359,13810.0,96.6,17196,5279,69.23,169.961667,185.360000,180.398333


#### **4. IGREA alignment**

IGREA measures global real economic activity at a monthly frequency, so it must be carried
onto the weekly calendar. Two things matter here: when each monthly value actually becomes
known, and how it is propagated between releases.

**Availability.** A month's value is not known on the first of that month — the index is
published with a delay. Each observation is therefore shifted forward by an approximate
release lag (two months, a deliberately conservative assumption to be refined) to obtain the
date from which it can legitimately be used.

**How.** Each Wednesday release is matched to the latest IGREA value already available at
that date, through a backward as-of join. Between two releases the last known value is simply
carried forward, which mirrors reality: a figure remains the current information until the
next one is published. Interpolating between consecutive months is avoided on purpose — it
would blend in the next month's value, which is not yet known, and reintroduce look-ahead
bias.


In [12]:
igrea = pd.read_csv("Dataset/FRED/index-of-global-real-economic-activity.csv",
                    parse_dates=["observation_date"])
igrea.columns = ["month", "igrea"]

# Shift by two months to approximate real availability (release lag).
igrea["available_from"] = igrea["month"] + pd.DateOffset(months=2)
igrea = igrea.sort_values("available_from")

# Carry the latest value available at each release date (no interpolation).
eia = pd.merge_asof(eia.sort_values("publication_date"),
                    igrea[["available_from", "igrea"]],
                    left_on="publication_date", right_on="available_from",
                    direction="backward").drop(columns="available_from")
eia.head()

,publication_date,week_ended,target_stock_change,crude_stocks,production,refinery_util,refiner_net_input,commercial_imports,wti_m1,gprd,gprd_act,gprd_threat,igrea_x,igrea_y
0,1982-08-25,1982-08-20,NaN,338764,NaN,NaN,11722,3459,NaN,NaN,NaN,NaN,-16.881230,-16.881230
1,1982-09-01,1982-08-27,-2626.0,336138,NaN,NaN,11918,3354,NaN,NaN,NaN,NaN,-41.180086,-41.180086
2,1982-09-29,1982-09-24,-552.0,335586,NaN,NaN,12375,3494,NaN,NaN,NaN,NaN,-41.180086,-41.180086
3,1982-10-06,1982-10-01,-800.0,334786,NaN,NaN,12303,2954,NaN,NaN,NaN,NaN,-43.524536,-43.524536
4,1982-10-13,1982-10-08,474.0,335260,NaN,NaN,12062,3008,NaN,NaN,NaN,NaN,-43.524536,-43.524536


#### **5. Check and export**

Before the dataset feeds the modelling stage, a few sanity checks confirm that sources
measured at different frequencies were aligned correctly. The row count and the span of
publication dates validate the reference calendar; the count of missing values per column
exposes coverage gaps and any misalignment — a column that is unexpectedly empty usually
points to a broken merge key.

Some gaps are expected rather than errors: `refinery_util` only starts in November 1990, and
the first `target_stock_change` is undefined by construction (first difference). The frame is
then written to CSV for the modelling stage.


In [11]:
print("Rows:", len(eia))
print("Span:", eia["publication_date"].min().date(), "->", eia["publication_date"].max().date())
print("\nMissing values:\n", eia.isna().sum())

eia.to_csv("H1_dataset_weekly.csv", index=False)


Rows: 2283
Span: 1982-08-25 -> 2026-07-01

Missing values:
 publication_date         0
week_ended               0
target_stock_change      1
crude_stocks             0
production              17
refinery_util          422
refiner_net_input        0
commercial_imports       0
wti_m1                  32
gprd                   117
gprd_act               117
gprd_threat            117
igrea                    0
dtype: int64


In [15]:
# First row where every column is populated, then drop everything before it.
start = eia.apply(lambda col: col.first_valid_index()).max()
eia = eia.loc[start:].reset_index(drop=True)
eia.head()

,publication_date,week_ended,target_stock_change,crude_stocks,production,refinery_util,refiner_net_input,commercial_imports,wti_m1,gprd,gprd_act,gprd_threat,igrea_x,igrea_y
0,1990-11-07,1990-11-02,4158.0,331392,7342.0,84.0,12898,4917,34.72,128.985000,103.858333,157.738333,-18.609531,-18.609531
1,1990-11-14,1990-11-09,1644.0,333036,7284.0,83.0,12684,4419,33.86,146.140000,92.320000,189.578333,-18.609531,-18.609531
2,1990-11-21,1990-11-16,-5743.0,327293,6910.0,84.0,12764,5637,31.41,150.308333,92.141667,209.955000,-18.609531,-18.609531
3,1990-11-28,1990-11-23,-3025.0,324268,7440.0,84.0,12779,5610,30.49,168.275000,142.723333,184.381667,-18.609531,-18.609531
4,1990-12-05,1990-11-30,2587.0,326855,7235.0,89.0,13711,4532,32.17,185.646667,115.896667,238.568333,-14.594048,-14.594048


In [17]:
eia.isna().sum()

publication_date       0
week_ended             0
target_stock_change    0
crude_stocks           0
production             0
refinery_util          0
refiner_net_input      0
commercial_imports     0
wti_m1                 2
gprd                   0
gprd_act               0
gprd_threat            0
igrea_x                0
igrea_y                0
dtype: int64